<a href="https://colab.research.google.com/github/SandroBolkvadze/Facial-Expression-Recognition-Kaggle-Competition/blob/main/face-expression-recognition-01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install dependencies

In [1]:
! pip install --upgrade kaggle --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 18.5 MB/s eta 0:00:00


# Imports

In [8]:
import pandas as pd
import numpy as np

import torch
import torchvision.transforms as transforms
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

# Use GPU
USE_GPU = True
device = 'cuda' if USE_GPU and torch.cuda.is_available() else 'cpu'
print('Device available:', device)

Device available: cuda


# Mount Google Drive

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Copy Access Token

In [10]:
! mkdir ~/.kaggle
! cp /content/drive/MyDrive/kaggle/access_token ~/.kaggle/access_token
! chmod 600 ~/.kaggle/access_token

# Download Kaggle Competition Data

In [11]:
! kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge

100% 285M/285M [00:02<00:00, 125MB/s]



# Unzip Kaggle Competition Data

In [12]:
! unzip challenges-in-representation-learning-facial-expression-recognition-challenge

Archive:  challenges-in-representation-learning-facial-expression-recognition-challenge.zip
  inflating: example_submission.csv  
  inflating: fer2013.tar.gz          
  inflating: icml_face_data.csv      
  inflating: test.csv                
  inflating: train.csv               


# Read & Transform Dataset

In [13]:

# Simple function for transforming 'pixels' elements
def pixels_to_image(pixels_str):
  pixels = np.array(pixels_str.split(), dtype=np.uint8)
  pixels = pixels.reshape(48, 48)
  return pixels

In [14]:
df = pd.read_csv('./icml_face_data.csv')

# Rename columns
df.columns = ['emotion', 'Usage', 'pixels']

# Choose Training part
df = df[df['Usage'] == 'Training']

# Drop 'Usage' column
df = df.drop(columns=['Usage'])

# Transform 'pixels' elements to numpy array in advance
df['pixels'] = df['pixels'].apply(pixels_to_image)


In [15]:
X = df.drop(columns=['emotion'])
y = df['emotion']

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

X shape: (28709, 1)
y shape: (28709,)


# Split Dataset

In [16]:
train_size = .7
val_size   = .15
test_size  = .15

# Split full dataset into train and val/test sets
X_train, X_val_test, y_train, y_val_test = train_test_split(
    X, y,
    test_size=(val_size+test_size)/(train_size+val_size+test_size),
    stratify=y,
)

# Split val/test dataset into 50/50 val and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_val_test, y_val_test,
    test_size=test_size/(val_size+test_size),
    stratify=y_val_test,
)

# Split full dataset into 70/15/15 train/val/test sets
print(f'X_train shape {X_train.shape} and y_train shape {y_train.shape}')
print(f'X_val shape {X_val.shape} and y_val shape {y_val.shape}')
print(f'X_test shape {X_test.shape} and y_test shape {y_test.shape}')

X_train shape (20096, 1) and y_train shape (20096,)
X_val shape (4306, 1) and y_val shape (4306,)
X_test shape (4307, 1) and y_test shape (4307,)


In [17]:
# Check if label distributions are same in train/val/test sets
y_train_dist = y_train.value_counts() / y_train.value_counts().sum()
y_val_dist   = y_val.value_counts()   / y_val.value_counts().sum()
y_test_dist  = y_test.value_counts()  / y_test.value_counts().sum()

# Should have roughly equal numbers across rows of 'emotions'
y_dist = pd.DataFrame({
    'y_train': y_train_dist,
    'y_val'  : y_val_dist,
    'y_test' : y_test_dist,
})

print(y_dist)

          y_train     y_val    y_test
emotion                              
3        0.251294  0.251277  0.251451
6        0.172920  0.173014  0.172974
4        0.168242  0.168137  0.168331
2        0.142715  0.142592  0.142791
0        0.139182  0.139108  0.139076
5        0.110470  0.110543  0.110286
1        0.015177  0.015327  0.015092


# Transform Raw Pixels

In [18]:
# Calculate mean pixel over whole train dataset
mean_pixel = X_train['pixels'].apply(np.mean).mean() / 255.0

# Transform 2d numpy pixel array
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(mean_pixel,), std=(1.0,)),
])

# Create Custom Image Dataset

In [19]:
class ImageDataset(Dataset):
  def __init__(self, X, y, transform=None):
    self.X = X
    self.y = torch.tensor(y.values)
    self.transform = transform

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, index):
    image = self.X.iloc[index, 0]
    label = self.y[index]

    if self.transform:
      image = self.transform(image)

    return image, label

# Hyperparameters

In [135]:
batch_size = 32
epochs = 10
lr = 1e-3

# Datasets & DataLoaders

In [136]:
train_dataset = ImageDataset(X_train, y_train, transform)
val_dataset   = ImageDataset(X_val, y_val, transform)
test_dataset  = ImageDataset(X_test, y_test, transform)

In [137]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

# Two Layer FC

In [138]:
class TwoLayerFC(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()

    # Initialize Layers
    self.fc1 = nn.Linear(input_size, hidden_size)
    self.fc2 = nn.Linear(hidden_size, output_size)
    self.relu = nn.ReLU()
    self.flatten = nn.Flatten()

    # Initialize weights & biases
    nn.init.kaiming_normal_(self.fc1.weight, nonlinearity='relu')
    nn.init.zeros_(self.fc1.bias)

    nn.init.xavier_normal_(self.fc2.weight)
    nn.init.zeros_(self.fc2.bias)


  def forward(self, x):
    # Flatten input
    x = self.flatten(x)

    # Apply Linear -> ReLU
    x = self.relu(self.fc1(x))

    # Apply Linear
    x = self.fc2(x)

    return x

model = TwoLayerFC(48*48, 256, 7).to(device)

# Loss & Optimizer

In [139]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=lr)

# Check Accuracy

In [140]:
def check_accuracy(loader, model):
  model.eval()

  num_correct = 0
  num_samples = 0
  with torch.no_grad():
    for t, (x, y) in enumerate(loader):
      x = x.to(device)
      y = y.to(device)

      scores = model(x)
      _, preds = scores.max(1)

      num_correct += (preds == y).sum()
      num_samples += y.shape[0]

  return num_correct / num_samples

# Train

In [141]:
def train(loader_train, loader_val, model, criterion, optimizer, print_every=100):

  for e in range(epochs):
    for t, (x, y) in enumerate(loader_train):
      x = x.to(device)
      y = y.to(device)

      model.train()

      # Calculate scores
      scores = model(x)

      # Calculate loss
      loss = criterion(scores, y)

      # Zero out paramater grads
      optimizer.zero_grad()

      # Backpropagation
      loss.backward()

      # Update model parameters
      optimizer.step()

      if t % print_every == 0:
        print('Epoch %d, Iteration %d, loss = %.4f' % (e, t, loss.item(),))
        print('train_acc: %.4f; ' % (check_accuracy(loader_train, model),), end='')

        if loader_val is not None:
            print('val_acc: %.4f' % (check_accuracy(loader_val, model),), end='')

        print('\n' + '-'*25)

# Check if Model can Overfit Small Data

In [142]:
small_data_size = 64

X_small_train = X_test[:small_data_size]
y_small_train = y_test[:small_data_size]

small_dataset = ImageDataset(X_small_train, y_small_train, transform)
small_loader  = DataLoader(small_dataset, batch_size=small_data_size, shuffle=True)

In [143]:
train(small_loader, None, model, criterion, optimizer, print_every=100000)

Epoch 0, Iteration 0, loss = 1.9288
train_acc: 0.5312; 
-------------------------
Epoch 1, Iteration 0, loss = 1.3664
train_acc: 0.7344; 
-------------------------
Epoch 2, Iteration 0, loss = 1.0319
train_acc: 0.8438; 
-------------------------
Epoch 3, Iteration 0, loss = 0.7816
train_acc: 0.9062; 
-------------------------
Epoch 4, Iteration 0, loss = 0.5878
train_acc: 1.0000; 
-------------------------
Epoch 5, Iteration 0, loss = 0.4415
train_acc: 1.0000; 
-------------------------
Epoch 6, Iteration 0, loss = 0.3334
train_acc: 1.0000; 
-------------------------
Epoch 7, Iteration 0, loss = 0.2512
train_acc: 1.0000; 
-------------------------
Epoch 8, Iteration 0, loss = 0.1887
train_acc: 1.0000; 
-------------------------
Epoch 9, Iteration 0, loss = 0.1419
train_acc: 1.0000; 
-------------------------
